# Colisão entre partículas
Nos notebooks anteriores, foi desenvolvido um modelo contendo múltiplas partículas confinadas em um recipiente, cada uma se movendo de forma independente e sofrendo colisões apenas com as paredes da caixa. Embora esse modelo já apresente características visuais semelhantes às de um gás, ele ainda não contempla as interações entre as próprias partículas.

Neste notebook, serão implementadas colisões entre partículas, permitindo que elas troquem quantidade de movimento durante os impactos. Para isso, será necessário detectar quando duas partículas entram em contato e aplicar uma regra de colisão compatível com o modelo físico adotado.

A inclusão dessas interações torna a simulação mais realista, aproximando-a do comportamento esperado de um gás ideal. Além disso, as colisões são fundamentais para o estudo de propriedades macroscópicas do sistema, como temperatura, pressão e distribuição de velocidades, que emergem do comportamento coletivo das partículas.


In [27]:
#Importa os módulos que serão utilizados
import pygame as pyg
import random as rd
import math as mt

In [28]:
def cria_particulas(particulas, num_particulas):
    """Função que cria as particulas, recebendo uma lista vazia e um número de particulas para a simulação. Ela retorna a lista com n particulas e as suas propriedades (velocidade e posição)."""
    
    #Criação do número dado de partículas
    for _ in range(num_particulas):
        #Angulo de direção alearório para cada partícula
        angulo = rd.uniform(0, 2 * mt.pi)
        
        #Direção x aleatoria
        x = rd.randint(
        caixa_x + raio,
        caixa_x + caixa_largura - raio
        )
        #Direção y aleatória
        y = rd.randint(
        caixa_y + raio,
        caixa_y + caixa_altura - raio
        )
        #Velocidades iguals, porém em direções aleatórias
        vx = velocidade*mt.cos(angulo)
        vy = velocidade*mt.sin(angulo)
        
        #Conjunto dentro de uma lista
        particulas.append(
        {
            "x": x,
            "y": y,
            "vx": vx,
            "vy": vy
        }
        )
    
    return particulas

In [29]:
def colisoes_parede(particulas):
    """Função que recebe a lista das particulas e inverte o sentido das suas velocidades quando elas colidem com a parede (colisões elasticas, retornando as velocidades invertidas das particulas."""
    
    for particula in particulas:
        #Criação das colisões para o teto e para o chão (lados direito e esquerdo)
    
        #Parede esquerda
        if particula["x"] <= caixa_x + raio:

            particula["x"] = caixa_x + raio

            particula["vx"] *= -1
        #parede direita
        if particula["x"] >= caixa_x + caixa_largura - raio:
            particula["x"] = caixa_x + caixa_largura - raio
            particula["vx"] *= -1
    
        #Teto superior
        if particula["y"] <= caixa_y + raio:
            particula["y"] = caixa_y + raio
            particula["vy"] *= -1
    
        #Teto inferior
        if particula["y"] >= caixa_y + caixa_altura - raio:
            particula["y"] = caixa_y + caixa_altura - raio
            particula["vy"] *= -1

In [30]:
def colisoes_particulas(particulas, raio):
    """Função que recebe a lista com as informações das partículas e detecta as colisões entre elas. Nesse código, as suas velocidades são invertidas (colisões elasticas) e um vetor de sobreposição normalizado separa elas imperceptívelmente para que uma nova colisão seja detectada (ler o markdown ao final do código)."""

    #Percorrer as partículas da lista e comparar as suas posições para as colisões
    for i in range(len(particulas)):
        #i+1, pois irá partir da próxima posição para vasculhar
        for j in range(i+1, len(particulas)):
            p1 = particulas[i]
            p2 = particulas[j]

            #distancia entre as partículas
            dx = p2["x"] - p1["x"]
            dy = p2["y"] - p1["y"]
            distancia = mt.sqrt((dx**2) + (dy**2))

            #caso elas se encostem (colisão), as velocidades são trocadas (colisão elastica)
            if distancia <= 2*raio and distancia > 0:
                p1["vx"], p2["vx"] = p2["vx"], p1["vx"]
                p1["vy"], p2["vy"] = p2["vy"], p1["vy"]

                #Separar minimamente as particulas para que colisões não sejam infinitas
                
                #normaliza os vetores de distancia entre as particulas
                nx = dx/distancia
                ny = dy/distancia

                #Corrige a sobreposição das partículas, ou seja, move cada partícula minimamente
                sobreposicao = 2*raio - distancia

                #Atualiza a posição das partículas, movendo-as minimamente para longe de si para que outra colisão não seja detectada
                p1["x"] -= nx * sobreposicao / 2
                p1["y"] -= ny * sobreposicao / 2
                p2["x"] += nx * sobreposicao / 2
                p2["y"] += ny * sobreposicao / 2

In [31]:
def mover_particulas(particulas):
    """Função que soma a velocidade de cada partícula à sua posição, dando a ideia de movimento."""
    
    #Atualiza a posição de cada particula (ideia de movimento)
    for particula in particulas:
        particula["x"] += particula["vx"]
        particula["y"] += particula["vy"]

In [32]:
def desenha_particulas(particulas, raio, tela):
    """Função que percorre as particulas e desenha as particulas na tela, recebendo a lista das particulas, o valor do seu raio e a tela da simulação."""
    
    #desenha as partículas na tela
    for particula in particulas:

        pyg.draw.circle(
            tela,
            (0, 0, 255),
            (
                int(particula["x"]),
                int(particula["y"])
            ),
            raio
        )

In [33]:
def desenha_caixa(tela, caixa_x, caixa_y, caixa_largura, caixa_altura):
    """Função que recebe as váriaveis das dimensões da caixa e retorna um desenho na tela da caixa com as dimensões proporcionadas."""
    
    #Desenho do recipiente
    pyg.draw.rect(
        tela,
        (20, 20, 20),
        (
            caixa_x,
            caixa_y,
            caixa_largura,
            caixa_altura
        ),
        width=2
    )
    

In [36]:
def texto_particulas(tela, fonte, particulas):
    """Função que recebe a fonte escolhida e a lista de partículas, retornando a escrita do número de particulas na tela."""
    
    #Mostra o número de partículas na tela
    texto = fonte.render(
    f"N = {len(particulas)}",
    True,
    (0, 0, 0)
    )
    tela.blit(texto, (620, 50))

In [37]:
pyg.init() #inicializa o pygame

tela = pyg.display.set_mode((800,600)) #Cria a janela e define o seu tamanho

#Título da janela
pyg.display.set_caption("Ideal Gas Lab")

#loop principal
rodando = True

#Definindo a variável clock como
clock = pyg.time.Clock()

#Criação do recipiente do gás
caixa_x = 25
caixa_y = 25

caixa_largura = 550
caixa_altura = 550

#Definindo a velocidade, a quantidade e o raio das partículas
velocidade = 3
raio = 10
num_particulas = 30

#Definindo a fonte que mostrará a mensagem na tela.
fonte = pyg.font.Font(None, 30)

#Criação das partículas
particulas = []

particulas = cria_particulas (particulas, num_particulas)

#Loop em que a simulação ocorrerá
while rodando:

    #Captura eventos
    for evento in pyg.event.get():
        if evento.type == pyg.QUIT:
            rodando = False
            
    #Deixa o fundo da tela cinza
    tela.fill((230, 230, 230))
    
    #movimenração das particulas (movimentação)
    mover_particulas(particulas)
    
    #Chama a função de colisões e muda o valor das variaveis de velocidade das particulas
    colisoes_parede(particulas)

    #detecta as colisões entre as particulas
    colisoes_particulas (particulas, raio)
    
    #desenha o recipiente das particulas
    desenha_caixa(tela, caixa_x, caixa_y, caixa_largura, caixa_altura)
    
    #desenha as partículas na tela
    desenha_particulas(particulas, raio, tela)

    #mostra o numero n de particulas
    texto_particulas(tela, fonte, particulas)
    
    
    #Limitando a atualização da tela
    #60 iterações por segundo (60 fps)
    clock.tick(60)
    
    #Atualiza a tela
    pyg.display.flip()



pyg.quit()

# Observação sobre o código:

Após a implementação da detecção de colisões, foi necessário desenvolver um mecanismo para evitar a sobreposição permanente das partículas. Quando duas partículas colidem, suas velocidades são trocadas e suas posições são corrigidas de modo que ambas permaneçam apenas em contato, sem ocupar a mesma região do espaço. Essa etapa é fundamental para garantir a estabilidade da simulação e evitar colisões repetidas causadas por erros numéricos decorrentes da discretização do tempo.